# 01 - Data audit

**Purpose.** Verify the metadata and audio before modelling. The audit *reports* problems; it never silently fixes, drops, duplicates or invents rows.

Run order: this notebook first. Everything is read-only with respect to the competition data.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
from src.config import load_config, repo_dir
from src.pipeline import get_context

# Dataset location: leave None to use the SHL_DATA_ROOT environment variable (or config.data_root),
# or set it here, e.g. DATA_ROOT = r"C:\Users\USERNAME\path\Dataset_Final"
DATA_ROOT = None
try:
    ctx = get_context(DATA_ROOT); cfg = ctx.cfg
    print("Data root:", ctx.paths.root)
except Exception as exc:
    ctx, cfg = None, load_config()
    print("Data root NOT configured ->", exc)
PENDING = "Results will be populated after local execution."
TABLES = repo_dir(cfg, "reports_dir") / "tables"
LOG_PATH = repo_dir(cfg, "experiments_dir") / "experiment_log.csv"
def have(*paths): return all(Path(p).exists() for p in paths)

In [ ]:
assert ctx is not None, "Set DATA_ROOT in the setup cell (or the SHL_DATA_ROOT environment variable) to your Dataset_Final folder, then re-run."

## 1. Load the three CSVs exactly as provided

In [ ]:
from src.data.loader import load_all_metadata
meta = load_all_metadata(ctx.paths)
for name, df in [("train", meta.train), ("test", meta.test), ("sample_submission", meta.sample)]:
    print(f"{name:18s} shape={df.shape} columns={list(df.columns)}")
    display(df.head(3))

## 2. Metadata audit (CSV only)

In [ ]:
from src.data.validator import audit_metadata
exp = cfg.get("expected", {})
report = audit_metadata(meta.train, meta.test, meta.sample, exp.get("train_rows"), exp.get("test_rows"))
for f in report.findings:
    print(f"[{f.level.upper():7s}] {f.code}: {f.message}")
print()
display(pd.DataFrame([report.stats["overlap"]]).T.rename(columns={0: "value"}))

### Interpretation rules
* `TEST_VS_SAMPLE_*` errors mean `test.csv` and `sample_submission.csv` do **not** describe the same set of files. Submission generation stops in that case (`scripts/make_submission.py`).
* `TRAIN_TEST_NAME_OVERLAP` is only a *warning*: train and test audio live in different folders, so identical names may be harmless re-numbering. The optional hash check below tells you whether the audio itself is identical (which would be leakage).
* Resolving the inconsistency is a **decision for the competition organisers / Kaggle data page**, not something code should guess.

## 3. Audio audit (needs the WAV folders)

In [ ]:
RUN_AUDIO_AUDIT = True       # reads WAV headers only (fast)
RUN_HASH_CHECK = False       # set True to MD5-compare same-named train/test files (reads all overlapping files)
if RUN_AUDIO_AUDIT and (ctx.paths.train_audio_dir.is_dir() or ctx.paths.test_audio_dir.is_dir()):
    from src.data.validator import audit_audio
    audit_audio(report, ctx.paths, meta.train, meta.test, meta.sample, hash_check=RUN_HASH_CHECK, progress=True)
    print("--- all findings after audio checks ---")
    for f in report.findings:
        print(f"[{f.level.upper():7s}] {f.code}: {f.message}")
    display(pd.json_normalize(report.stats["audio"], sep=".").T.rename(columns={0: "value"}))
else:
    print("Audio folders not found.", PENDING)

## 4. Save the audit

In [ ]:
TABLES.mkdir(parents=True, exist_ok=True)
report.to_json(TABLES / "dataset_audit.json")
(TABLES / "dataset_audit.md").write_text(report.to_markdown(), encoding="utf-8")
if report.inventory is not None:
    report.inventory.drop(columns=["path"]).to_csv(TABLES / "audio_inventory.csv", index=False)
print("saved to", TABLES)

## 5. Training label distribution

In [ ]:
from src.reporting import plot_label_distribution
y = pd.to_numeric(meta.train["label"], errors="coerce")
display(y.describe().to_frame().T)
plot_label_distribution(y); plt.show()

## 6. What to do about the inconsistency
1. Re-download / re-check the Kaggle *Data* tab: which CSV defines the scored test set?
2. Run the hash check above to rule out train/test audio duplication.
3. Do **not** edit the CSVs. If `sample_submission.csv` is the authoritative list, process it explicitly with `--manifest sample_submission` (audio is then read from `test/` and every listed file must exist).
4. Record the decision in `reports/final_report.md`.